# Agent 1 — AI Summarization

Reads all validation runs with ai_enrichment_status = PENDING,
assembles an evidence bundle for each run, calls the AI model,
and writes the enrichment back to the results tables.

**Runs asynchronously after the validation agent.**
**Never blocks or modifies deterministic check results.**

Reads from:
  uc1_validation_run_b
  uc1_validation_results_b
  uc1_baseline_metrics_b
  uc1_table_registry_b

Updates:
  uc1_validation_results_b  (ai_summary, ai_recommended_action,
                           ai_failure_category, ai_confidence_score,
                           ai_model_version, ai_invocation_id,
                           ai_enriched_timestamp)

  uc1_validation_run_b      (ai_enrichment_status,
                           ai_enrichment_timestamp)

In [0]:
import requests
import json
import uuid
import os
from datetime import datetime
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Catalog and Schema ─────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# ── Table References ───────────────────────────────────────────────────────

VALIDATION_RUN     = f"{CATALOG}.{SCHEMA}.uc1_validation_run_b"
VALIDATION_RESULTS = f"{CATALOG}.{SCHEMA}.uc1_validation_results_b"
BASELINE_METRICS   = f"{CATALOG}.{SCHEMA}.uc1_baseline_metrics_b"
TABLE_REGISTRY     = f"{CATALOG}.{SCHEMA}.uc1_table_registry_b"

# ── Model Configuration ────────────────────────────────────────────────────

MODEL_ENDPOINT = "databricks-gpt-oss-20b"
MODEL_VERSION  = "gpt-oss-20b-080525"   # actual model version from response
MAX_TOKENS     = 4096                   # increased for reasoning model
TEMPERATURE    = 0.1

# ── Workspace Connection ───────────────────────────────────────────────────

WORKSPACE_HOST  = spark.conf.get("spark.databricks.workspaceUrl")
WORKSPACE_TOKEN = (
    dbutils.notebook.entry_point
           .getDbutils().notebook().getContext()
           .apiToken().get()
)

HEADERS = {
    "Authorization": f"Bearer {WORKSPACE_TOKEN}",
    "Content-Type" : "application/json"
}

MODEL_URL = (
    f"https://{WORKSPACE_HOST}"
    f"/serving-endpoints/{MODEL_ENDPOINT}/invocations"
)

VALID_FAILURE_CATEGORIES = {
    "SCHEMA_ISSUE",
    "VOLUME_ISSUE",
    "QUALITY_ISSUE",
    "PIPELINE_ISSUE",
    "COMBINED_ISSUE"
}

print("=" * 60)
print("AGENT 1 — AI SUMMARIZATION")
print("=" * 60)
print(f"Workspace host : {WORKSPACE_HOST}")
print(f"Model endpoint : {MODEL_ENDPOINT}")
print(f"Model version  : {MODEL_VERSION}")
print(f"Max tokens     : {MAX_TOKENS}")
print(f"Token length   : {len(WORKSPACE_TOKEN)}")
print(f"Started at     : {datetime.now()}")
print("=" * 60)

AGENT 1 — AI SUMMARIZATION
Workspace host : adb-8005091109336306.6.azuredatabricks.net
Model endpoint : databricks-gpt-oss-20b
Model version  : gpt-oss-20b-080525
Max tokens     : 4096
Token length   : 38
Started at     : 2026-11-11 16:33:09.165827


## Step 1: Load All Pending Validation Runs

In [0]:
from pyspark.sql import functions as F

def load_pending_runs() -> list:
    """
    Loads all validation runs where AI enrichment
    is pending. Returns list of dicts ordered by
    business_date and table name.
    """
    df = (
        spark.table(VALIDATION_RUN)
             .filter(F.col("ai_enrichment_status") == "PENDING")
             .orderBy("business_date", "fully_qualified_name")
    )

    rows = [row.asDict() for row in df.collect()]

    print(f"Pending runs found: {len(rows)}")

    for r in rows:
        print(
            f"  {r['business_date']}  "
            f"{r['fully_qualified_name'].split('.')[-1]:<35}  "
            f"Status: {r['overall_status']}"
        )

    return rows


pending_runs = load_pending_runs()

# ── Graceful handling when no pending runs exist ───────────────────────────
# DO NOT use dbutils.notebook.exit() here.
# In a Databricks Workflow context, notebook.exit() with a message
# can cause the downstream task dependency evaluation to behave
# unexpectedly.
# Instead we set a flag and let remaining cells handle it gracefully.

if not pending_runs:
    print()
    print("No pending runs found. Nothing to process.")
    print("This is expected when:")
    print("  - All runs passed (AI enrichment not required)")
    print("  - AI was already completed in a previous execution")
    print("  - Validation agent has not run yet for today")
    print()
    print("Notebook will complete gracefully without processing.")
    NO_PENDING_RUNS = True
else:
    NO_PENDING_RUNS = False
    print(f"\nProceeding to process {len(pending_runs)} run(s).")

Pending runs found: 1
  2026-10-14  bronze_barrows_campaign              Status: CRITICAL

Proceeding to process 1 run(s).


## Step 2: Evidence Bundle Assembly

In [0]:
def load_failed_checks(validation_run_id: str) -> list:
    """
    Loads all WARNING and CRITICAL check results
    for a given validation run.
    """
    df = (
        spark.table(VALIDATION_RESULTS)
             .filter(
                 (F.col("validation_run_id") == validation_run_id) &
                 (F.col("check_status").isin(["WARNING", "CRITICAL"]))
             )
             .orderBy(
                 F.when(F.col("check_status") == "CRITICAL", 1)
                  .otherwise(2),
                 "check_type"
             )
    )
    return [row.asDict() for row in df.collect()]


def load_baseline_context(
    fqn            : str,
    failed_checks  : list
) -> list:
    """
    Loads baseline metrics for each failed check.
    Provides the AI with historical context.
    """
    baselines = []

    for check in failed_checks:
        check_type  = check["check_type"]
        column_name = check["column_name"] or "__table__"

        df = (
            spark.table(BASELINE_METRICS)
                 .filter(
                     (F.col("fully_qualified_name") == fqn) &
                     (F.col("check_type") == check_type) &
                     (F.col("column_name") == column_name)
                 )
                 .orderBy(F.col("baseline_date").desc())
                 .limit(1)
        )

        rows = [row.asDict() for row in df.collect()]
        if rows:
            b = rows[0]
            baselines.append({
                "check_type"      : check_type,
                "column_name"     : column_name,
                "metric_name"     : b.get("metric_name"),
                "baseline_mean"   : b.get("mean_value"),
                "baseline_stddev" : b.get("stddev_value"),
                "baseline_min"    : b.get("min_value"),
                "baseline_max"    : b.get("max_value"),
                "sample_count"    : b.get("sample_count"),
                "baseline_status" : b.get("baseline_status"),
                "window_days"     : b.get("baseline_window_days")
            })

    return baselines


def load_table_context(fqn: str) -> dict:
    """
    Loads table registration metadata for context.
    """
    df = (
        spark.table(TABLE_REGISTRY)
             .filter(F.col("fully_qualified_name") == fqn)
             .limit(1)
    )
    rows = [row.asDict() for row in df.collect()]
    if not rows:
        return {}

    t = rows[0]
    return {
        "source_type"                    : t.get("source_type"),
        "ingestion_pattern"              : t.get("ingestion_pattern"),
        "ingestion_frequency"            : t.get("ingestion_frequency"),
        "partition_column"               : t.get("partition_column"),
        "business_date_column"           : t.get("business_date_column"),
        "expected_arrival_window_minutes": t.get("expected_arrival_window_minutes")
    }


def assemble_evidence_bundle(
    run_record    : dict,
    failed_checks : list,
    baselines     : list,
    table_context : dict
) -> dict:
    """
    Assembles the complete evidence bundle for one validation run.
    This is the structured input to the AI model.
    """
    fqn = run_record["fully_qualified_name"]

    # Parse check_detail JSON for each failed check
    cleaned_checks = []
    for check in failed_checks:
        detail = {}
        if check.get("check_detail"):
            try:
                detail = json.loads(check["check_detail"])
            except (json.JSONDecodeError, TypeError):
                detail = {}

        cleaned_checks.append({
            "check_type"        : check["check_type"],
            "check_level"       : check["check_level"],
            "column_name"       : check["column_name"],
            "check_status"      : check["check_status"],
            "observed_value"    : check["observed_value"],
            "baseline_value"    : check["baseline_value"],
            "deviation_pct"     : check["deviation_percentage"],
            "threshold_warning" : check["threshold_warning"],
            "threshold_critical": check["threshold_critical"],
            "check_detail"      : detail
        })

    return {
        "context": {
            "validation_run_id"  : run_record["validation_run_id"],
            "fully_qualified_name": fqn,
            "business_date"      : str(run_record["business_date"]),
            "source_type"        : table_context.get("source_type"),
            "ingestion_pattern"  : table_context.get("ingestion_pattern"),
            "ingestion_frequency": table_context.get("ingestion_frequency"),
            "partition_column"   : table_context.get("partition_column"),
            "expected_arrival_window_minutes": table_context.get(
                "expected_arrival_window_minutes"
            )
        },
        "run_summary": {
            "overall_status"       : run_record["overall_status"],
            "total_checks_run"     : run_record["total_checks_run"],
            "total_checks_passed"  : run_record["total_checks_passed"],
            "total_checks_warned"  : run_record["total_checks_warned"],
            "total_checks_critical": run_record["total_checks_critical"]
        },
        "failed_checks"   : cleaned_checks,
        "baseline_context": baselines
    }


print("Evidence bundle functions loaded.")

Evidence bundle functions loaded.


## Step 3: Prompt Builder

In [0]:
def build_prompt(evidence_bundle: dict) -> str:

    fqn           = evidence_bundle["context"]["fully_qualified_name"]
    table_name    = fqn.split(".")[-1]
    business_date = evidence_bundle["context"]["business_date"]
    source_type   = evidence_bundle["context"]["source_type"]
    pattern       = evidence_bundle["context"]["ingestion_pattern"]
    frequency     = evidence_bundle["context"]["ingestion_frequency"]
    overall       = evidence_bundle["run_summary"]["overall_status"]
    failed_checks = evidence_bundle["failed_checks"]
    baselines     = evidence_bundle["baseline_context"]

    checks_text = ""
    for i, check in enumerate(failed_checks, 1):
        checks_text += f"""
  Check {i}:
    Table       : {table_name}
    Type        : {check['check_type']}
    Level       : {check['check_level']}
    Column      : {check['column_name'] or 'Table-level'}
    Status      : {check['check_status']}
    Observed    : {check['observed_value']}
    Baseline    : {check['baseline_value']}
    Deviation   : {check['deviation_pct']}%
    Warning at  : {check['threshold_warning']}
    Critical at : {check['threshold_critical']}
    Detail      : {json.dumps(check['check_detail'], indent=6)}
"""

    baseline_text = ""
    for b in baselines:
        baseline_text += f"""
  {b['check_type']} on {table_name}.{b['column_name']}:
    Baseline mean   : {b['baseline_mean']}
    Baseline stddev : {b['baseline_stddev']}
    Sample days     : {b['sample_count']}
    Window days     : {b['window_days']}
"""

    prompt = f"""You are a senior data quality engineer analyzing
automated validation results for a retail data platform.

CRITICAL SCOPE CONSTRAINT:
You are analyzing ONE table only: {table_name}
Business date: {business_date}
Do not reference any other table in your response.
Every column you mention must belong to {table_name}.

TABLE INFORMATION:
  Table name         : {table_name}
  Full name          : {fqn}
  Business date      : {business_date}
  Source type        : {source_type}
  Ingestion pattern  : {pattern}
  Ingestion frequency: {frequency}
  Overall status     : {overall}

FAILED CHECKS IN {table_name} ON {business_date}:
({len(failed_checks)} check(s) failed)
{checks_text}

HISTORICAL BASELINE FOR {table_name}:
{baseline_text}

Produce a JSON response with exactly these four fields.
Every sentence in your response must refer to {table_name}
and business date {business_date} only.

{{
  "ai_summary": "2-3 sentences describing what went wrong
                  in {table_name} on {business_date}.
                  Name specific columns and exact numbers.",

  "ai_recommended_action": "1-2 specific steps the data
                             engineering team should take
                             for {table_name} on {business_date}.
                             Be precise about what to check.",

  "ai_failure_category": "One of: SCHEMA_ISSUE, VOLUME_ISSUE,
                           QUALITY_ISSUE, PIPELINE_ISSUE,
                           COMBINED_ISSUE",

  "ai_confidence_score": 0.0 to 1.0
}}

Respond with valid JSON only. No markdown. No text outside
the JSON object. Every reference must be to {table_name}."""

    return prompt


print("Prompt builder loaded.")

Prompt builder loaded.


## Step 4: AI Model Caller and Response Parser

In [0]:
def extract_content_from_response(response_data: dict) -> str:
    """
    Extracts text content from model response.
    Handles the gpt-oss-20b reasoning block format specifically.
    """
    choices = response_data.get("choices", [])
    if not choices:
        logger.warning("No choices in response")
        return ""

    first_choice  = choices[0]
    finish_reason = first_choice.get("finish_reason", "")
    message       = first_choice.get("message", {})
    content       = message.get("content", "")

    logger.info(f"finish_reason: {finish_reason}")
    logger.info(f"content type: {type(content)}")

    if finish_reason == "length":
        logger.warning(
            "Model response was cut off due to token limit. "
            "Consider increasing MAX_TOKENS further."
        )

    # Simple string content
    if isinstance(content, str):
        return content

    # List of content blocks
    if isinstance(content, list):
        text_parts     = []
        reasoning_text = []

        for block in content:
            if not isinstance(block, dict):
                if isinstance(block, str):
                    text_parts.append(block)
                continue

            block_type = block.get("type", "")

            # Direct text block — this is the actual answer
            if block_type == "text":
                text = block.get("text", "")
                if text:
                    text_parts.append(text)
                    logger.info(f"Found text block: {text[:100]}")

            # Output text block
            elif block_type == "output_text":
                text = block.get("text", "")
                if text:
                    text_parts.append(text)

            # Reasoning block — extract summary as fallback
            elif block_type == "reasoning":
                summary = block.get("summary", [])
                for s in summary:
                    if isinstance(s, dict):
                        s_text = s.get("text", "")
                        if s_text:
                            reasoning_text.append(s_text)

        # Prefer direct text blocks
        if text_parts:
            return " ".join(text_parts)

        # Fall back to reasoning text if no direct answer found
        if reasoning_text:
            logger.warning(
                "Only reasoning blocks found, no direct text block. "
                "The model may need more tokens to complete its response."
            )
            return " ".join(reasoning_text)

    # Fallback: choices[0].text
    text = first_choice.get("text", "")
    if text:
        return text

    logger.warning(f"Could not extract content from response")
    return ""


def call_ai_model(prompt: str, invocation_id: str) -> dict:
    """
    Calls the AI model and returns parsed response.
    """
    payload = {
        "messages": [
            {
                "role"   : "system",
                "content": (
                    "You are a data quality engineer for a retail "
                    "data platform. Always respond with valid JSON "
                    "only. Never include markdown, code fences, or "
                    "any text outside the JSON object."
                )
            },
            {
                "role"   : "user",
                "content": prompt
            }
        ],
        "max_tokens" : MAX_TOKENS,
        "temperature": TEMPERATURE
    }

    try:
        logger.info(f"Calling model: {MODEL_URL}")
        response = requests.post(
            MODEL_URL,
            headers=HEADERS,
            json=payload,
            timeout=180     # 3 minutes for reasoning model
        )

        logger.info(f"Response status: {response.status_code}")

        if response.status_code != 200:
            error_msg = (
                f"HTTP {response.status_code}: "
                f"{response.text[:500]}"
            )
            logger.error(f"Model call failed: {error_msg}")
            return {
                "success"     : False,
                "error"       : error_msg,
                "raw_response": response.text
            }

        response_data = response.json()

        # Extract content using robust extractor
        raw_content = extract_content_from_response(response_data)
        logger.info(f"Extracted content: {str(raw_content)[:300]}")

        # Ensure string
        if not isinstance(raw_content, str):
            raw_content = str(raw_content)

        if not raw_content.strip():
            return {
                "success"     : False,
                "error"       : "Empty content extracted from model response",
                "raw_response": str(response_data)
            }

        # Parse JSON from content
        parsed = parse_ai_response(raw_content)

        if parsed["success"]:
            return {
                "success"              : True,
                "ai_summary"           : parsed["ai_summary"],
                "ai_recommended_action": parsed["ai_recommended_action"],
                "ai_failure_category"  : parsed["ai_failure_category"],
                "ai_confidence_score"  : parsed["ai_confidence_score"],
                "raw_response"         : raw_content
            }
        else:
            return {
                "success"     : False,
                "error"       : parsed["error"],
                "raw_response": raw_content
            }

    except requests.exceptions.Timeout:
        logger.error("Model call timed out after 180 seconds.")
        return {
            "success"     : False,
            "error"       : "Model call timed out after 180 seconds",
            "raw_response": None
        }
    except Exception as e:
        import traceback
        logger.error(f"Unexpected error: {str(e)}")
        traceback.print_exc()
        return {
            "success"     : False,
            "error"       : str(e),
            "raw_response": None
        }


def parse_ai_response(raw_content: str) -> dict:
    """
    Parses the raw model response string into structured fields.
    """
    if not raw_content or not raw_content.strip():
        return {
            "success": False,
            "error"  : "Empty response from model"
        }

    content = raw_content.strip()

    # Remove markdown code blocks
    if "```json" in content:
        parts = content.split("```json")
        if len(parts) > 1:
            content = parts[1]
            if "```" in content:
                content = content.split("```")[0]
    elif "```" in content:
        parts = content.split("```")
        if len(parts) >= 3:
            content = parts[1]
        elif len(parts) == 2:
            content = parts[1]

    content = content.strip()

    # Find JSON object
    start = content.find("{")
    end   = content.rfind("}") + 1

    if start == -1 or end == 0:
        return {
            "success": False,
            "error"  : (
                f"No JSON object found. "
                f"Content preview: {content[:300]}"
            )
        }

    json_str = content[start:end]

    try:
        parsed = json.loads(json_str)
    except json.JSONDecodeError as e:
        return {
            "success": False,
            "error"  : (
                f"JSON parse error: {str(e)}. "
                f"JSON preview: {json_str[:300]}"
            )
        }

    # Validate required fields
    required_fields = [
        "ai_summary",
        "ai_recommended_action",
        "ai_failure_category",
        "ai_confidence_score"
    ]

    missing = [f for f in required_fields if f not in parsed]
    if missing:
        return {
            "success": False,
            "error"  : f"Missing required fields: {missing}"
        }

    # Validate category
    category = str(parsed.get("ai_failure_category", "")).upper().strip()
    if category not in VALID_FAILURE_CATEGORIES:
        logger.warning(
            f"Invalid category '{category}', defaulting to COMBINED_ISSUE"
        )
        category = "COMBINED_ISSUE"

    # Validate confidence
    try:
        confidence = float(parsed.get("ai_confidence_score", 0.5))
        confidence = max(0.0, min(1.0, confidence))
    except (TypeError, ValueError):
        confidence = 0.5

    summary = str(parsed.get("ai_summary", "")).strip()
    action  = str(parsed.get("ai_recommended_action", "")).strip()

    if not summary:
        return {
            "success": False,
            "error"  : "ai_summary field is empty"
        }

    return {
        "success"              : True,
        "ai_summary"           : summary,
        "ai_recommended_action": action,
        "ai_failure_category"  : category,
        "ai_confidence_score"  : confidence
    }


print("AI model caller and parser loaded.")

AI model caller and parser loaded.


## Step 5: Write AI Enrichment Back to Tables

In [0]:
def write_ai_enrichment_to_results(
    validation_run_id  : str,
    ai_summary         : str,
    ai_recommended_action: str,
    ai_failure_category: str,
    ai_confidence_score: float,
    ai_invocation_id   : str,
    enriched_timestamp : datetime
) -> bool:
    """
    Updates all failed check rows for a validation run
    with the AI enrichment fields.

    Writes the same AI summary to every failed check row
    so results are self-contained without needing a join.
    """
    try:
        # Escape single quotes in AI text to prevent SQL injection
        safe_summary = ai_summary.replace("'", "\\'")
        safe_action  = ai_recommended_action.replace("'", "\\'")

        spark.sql(f"""
            UPDATE {VALIDATION_RESULTS}
            SET
                ai_summary              = '{safe_summary}',
                ai_recommended_action   = '{safe_action}',
                ai_failure_category     = '{ai_failure_category}',
                ai_confidence_score     = {ai_confidence_score},
                ai_model_version        = '{MODEL_VERSION}',
                ai_invocation_id        = '{ai_invocation_id}',
                ai_enriched_timestamp   = '{enriched_timestamp}'
            WHERE validation_run_id     = '{validation_run_id}'
              AND check_status IN ('WARNING', 'CRITICAL')
        """)

        logger.info(
            f"AI enrichment written for run {validation_run_id}"
        )
        return True

    except Exception as e:
        logger.error(
            f"Failed to write AI enrichment for run "
            f"{validation_run_id}: {str(e)}"
        )
        return False


def update_run_status(
    validation_run_id  : str,
    ai_status          : str,
    enriched_timestamp : datetime
) -> bool:
    """
    Updates the validation run record with the
    AI enrichment completion status.
    """
    try:
        spark.sql(f"""
            UPDATE {VALIDATION_RUN}
            SET
                ai_enrichment_status    = '{ai_status}',
                ai_enrichment_timestamp = '{enriched_timestamp}'
            WHERE validation_run_id     = '{validation_run_id}'
        """)

        logger.info(
            f"Run {validation_run_id} status updated to {ai_status}"
        )
        return True

    except Exception as e:
        logger.error(
            f"Failed to update run status for "
            f"{validation_run_id}: {str(e)}"
        )
        return False


print("Results writer functions loaded.")

Results writer functions loaded.


## Step 6: Process All Pending Runs

In [0]:
runs_processed   = 0
runs_succeeded   = 0
runs_failed      = 0
invocation_log   = []

# ── Check flag set in Cell 3 ───────────────────────────────────────────────

if NO_PENDING_RUNS:
    print("Skipping processing — no pending runs found.")
    print("AI summarization complete with nothing to process.")

else:
    print(f"Processing {len(pending_runs)} pending run(s)...\n")

    for run_record in pending_runs:

        run_id   = run_record["validation_run_id"]
        fqn      = run_record["fully_qualified_name"]
        biz_date = run_record["business_date"]

        print(f"{'='*60}")
        print(f"Run ID       : {run_id}")
        print(f"Table        : {fqn}")
        print(f"Date         : {biz_date}")
        print(f"{'='*60}")

        invocation_id = str(uuid.uuid4())

        try:
            # ── Step A: Load failed checks ─────────────────────────────────
            failed_checks = load_failed_checks(run_id)
            print(f"Failed checks loaded   : {len(failed_checks)}")

            if not failed_checks:
                print("No failed checks found. Skipping AI call.")
                update_run_status(run_id, "NOT_REQUIRED", datetime.now())
                runs_processed += 1
                continue

            # ── Step B: Load baseline context ──────────────────────────────
            baselines = load_baseline_context(fqn, failed_checks)
            print(f"Baseline records loaded: {len(baselines)}")

            # ── Step C: Load table context ─────────────────────────────────
            table_context = load_table_context(fqn)
            print(
                f"Table context loaded   : "
                f"{table_context.get('source_type')} / "
                f"{table_context.get('ingestion_pattern')}"
            )

            # ── Step D: Assemble evidence bundle ───────────────────────────
            evidence_bundle = assemble_evidence_bundle(
                run_record,
                failed_checks,
                baselines,
                table_context
            )
            print(f"Evidence bundle assembled.")

            # ── Step E: Build prompt ───────────────────────────────────────
            prompt        = build_prompt(evidence_bundle)
            prompt_length = len(prompt)
            print(f"Prompt length          : {prompt_length} characters")

            # ── Step F: Call AI model ──────────────────────────────────────
            print(f"Calling model          : {MODEL_ENDPOINT}")
            ai_result = call_ai_model(prompt, invocation_id)

            if ai_result["success"]:
                print(f"Model call succeeded.")
                print(f"Category               : {ai_result['ai_failure_category']}")
                print(f"Confidence             : {ai_result['ai_confidence_score']}")
                print(f"Summary preview        : {ai_result['ai_summary'][:100]}...")

                # ── Step G: Write enrichment to results table ──────────────
                enriched_at   = datetime.now()

                write_success = write_ai_enrichment_to_results(
                    validation_run_id      = run_id,
                    ai_summary             = ai_result["ai_summary"],
                    ai_recommended_action  = ai_result["ai_recommended_action"],
                    ai_failure_category    = ai_result["ai_failure_category"],
                    ai_confidence_score    = ai_result["ai_confidence_score"],
                    ai_invocation_id       = invocation_id,
                    enriched_timestamp     = enriched_at
                )

                # ── Step H: Update run status ──────────────────────────────
                if write_success:
                    update_run_status(run_id, "COMPLETED", enriched_at)
                    runs_succeeded += 1
                    print(f"Status                 : COMPLETED ✅")
                else:
                    update_run_status(run_id, "FAILED", datetime.now())
                    runs_failed += 1
                    print(f"Status                 : FAILED (write error) ❌")

            else:
                print(f"Model call failed: {ai_result.get('error')}")
                update_run_status(run_id, "FAILED", datetime.now())
                runs_failed += 1

            # ── Log invocation ─────────────────────────────────────────────
            invocation_log.append({
                "run_id"       : run_id,
                "fqn"          : fqn,
                "business_date": str(biz_date),
                "invocation_id": invocation_id,
                "success"      : ai_result["success"],
                "category"     : ai_result.get("ai_failure_category"),
                "confidence"   : ai_result.get("ai_confidence_score"),
                "error"        : ai_result.get("error")
            })

        except Exception as e:
            print(f"UNEXPECTED ERROR: {str(e)}")
            import traceback
            traceback.print_exc()
            update_run_status(run_id, "FAILED", datetime.now())
            runs_failed += 1

        finally:
            runs_processed += 1
            print()

    print(f"Processing complete.")
    print(f"  Processed  : {runs_processed}")
    print(f"  Succeeded  : {runs_succeeded}")
    print(f"  Failed     : {runs_failed}")

Processing 1 pending run(s)...

Run ID       : ea02b5d9-db2a-45d6-9967-2721a1124b01
Table        : comms_media_prd.exposure_data_asset.bronze_barrows_campaign
Date         : 2026-10-14
Failed checks loaded   : 3
Baseline records loaded: 3


INFO:__main__:Calling model: https://adb-8005091109336306.6.azuredatabricks.net/serving-endpoints/databricks-gpt-oss-20b/invocations


Table context loaded   : FILE / BRONZE
Evidence bundle assembled.
Prompt length          : 3811 characters
Calling model          : databricks-gpt-oss-20b


INFO:__main__:Response status: 200
INFO:__main__:finish_reason: stop
INFO:__main__:content type: <class 'str'>
INFO:__main__:Extracted content: {"ai_summary":"On 2026-10-14 bronze_barrows_campaign experienced a complete loss of data: the partnerId column showed a 100.0% decrease in distinct values (current 0 vs baseline 0.14285714285714285), the id column also dropped 100.0% (current 0 vs baseline 1.0), and the table volume anomaly indicate


Model call succeeded.
Category               : PIPELINE_ISSUE
Confidence             : 0.9
Summary preview        : On 2026-10-14 bronze_barrows_campaign experienced a complete loss of data: the partnerId column show...


INFO:__main__:AI enrichment written for run ea02b5d9-db2a-45d6-9967-2721a1124b01
INFO:__main__:Run ea02b5d9-db2a-45d6-9967-2721a1124b01 status updated to COMPLETED


Status                 : COMPLETED ✅

Processing complete.
  Processed  : 1
  Succeeded  : 1
  Failed     : 0


## Step 7: Verify AI Enrichment Results

In [0]:
print("=" * 60)
print("AI ENRICHMENT VERIFICATION")
print("=" * 60)

# Run status summary
print("\n1. Validation Run AI Status:\n")
spark.sql(f"""
    SELECT
        business_date,
        fully_qualified_name,
        overall_status,
        ai_enrichment_status,
        ai_enrichment_timestamp
    FROM {VALIDATION_RUN}
    ORDER BY business_date, fully_qualified_name
""").show(20, truncate=False)

# AI enriched results
print("\n2. AI Enriched Check Results:\n")
spark.sql(f"""
    SELECT
        r.business_date,
        r.fully_qualified_name,
        r.check_type,
        r.column_name,
        r.check_status,
        r.ai_failure_category,
        ROUND(r.ai_confidence_score, 2) AS confidence,
        r.ai_summary,
        r.ai_recommended_action
    FROM {VALIDATION_RESULTS} r
    WHERE r.ai_summary IS NOT NULL
    ORDER BY
        r.business_date,
        r.fully_qualified_name,
        r.check_type
""").show(50, truncate=False)

# Invocation summary
print("\n3. AI Invocation Log:\n")
for entry in invocation_log:
    print(f"  Date     : {entry['business_date']}")
    print(f"  Table    : {entry['fqn'].split('.')[-1]}")
    print(f"  Category : {entry['category']}")
    print(f"  Confidence: {entry['confidence']}")
    print(f"  Success  : {entry['success']}")
    if not entry['success']:
        print(f"  Error    : {entry['error']}")
    print()

AI ENRICHMENT VERIFICATION

1. Validation Run AI Status:

+-------------+-----------------------------------------------------------+--------------+--------------------+--------------------------+
|business_date|fully_qualified_name                                       |overall_status|ai_enrichment_status|ai_enrichment_timestamp   |
+-------------+-----------------------------------------------------------+--------------+--------------------+--------------------------+
|2026-10-14   |comms_media_prd.exposure_data_asset.bronze_barrows_campaign|CRITICAL      |COMPLETED           |2026-11-11 16:33:19.855046|
|2026-10-17   |comms_media_prd.exposure_data_asset.bronze_barrows_campaign|CRITICAL      |COMPLETED           |2026-11-11 16:30:10.039769|
|2026-10-19   |comms_media_prd.exposure_data_asset.bronze_barrows_campaign|CRITICAL      |COMPLETED           |2026-11-11 16:30:19.515222|
+-------------+-----------------------------------------------------------+--------------+------------------

## Final Summary

In [0]:
print("\n" + "=" * 60)
print("AI SUMMARIZATION COMPLETE")
print("=" * 60)

if NO_PENDING_RUNS:
    print("Runs processed   : 0")
    print("Runs succeeded   : 0")
    print("Runs failed      : 0")
    print("Reason           : No pending runs found")
else:
    print(f"Runs processed   : {runs_processed}")
    print(f"Runs succeeded   : {runs_succeeded}")
    print(f"Runs failed      : {runs_failed}")

    if runs_failed > 0:
        print()
        print(
            f"WARNING: {runs_failed} run(s) failed AI enrichment. "
            f"Deterministic results are unaffected. "
            f"Re-run this notebook to retry failed runs."
        )

    if runs_succeeded > 0:
        print()
        print(
            f"SUCCESS: {runs_succeeded} run(s) enriched with AI summaries."
        )

print(f"Model used       : {MODEL_ENDPOINT}")
print(f"Completed at     : {datetime.now()}")
print("=" * 60)


AI SUMMARIZATION COMPLETE
Runs processed   : 1
Runs succeeded   : 1
Runs failed      : 0

SUCCESS: 1 run(s) enriched with AI summaries.
Model used       : databricks-gpt-oss-20b
Completed at     : 2026-11-11 16:33:26.409361
